# Teste 3: Análise de Re-ranking

Compara a obtenção de resultados (retrieval) COM e SEM re-ranking utilizando um modelo CrossEncoder.

Analisando:

**Nota:** Este notebook inclui geração de texto com SLM, o que exige mais tempo de processamento e download.

## 1. Instalar as dependências

In [1]:
!pip install -q langchain langchain-community langchain-chroma langchain-experimental
!pip install -q langchain-huggingface sentence-transformers
!pip install -q chromadb pymupdf rank-bm25 transformers accelerate

In [2]:
# faz o download do livro pelo notebook python do Colab
!wget https://domainpublic.wordpress.com/wp-content/uploads/2022/10/jk_couto_2ed.pdf

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_experimental.text_splitter import SemanticChunker
import torch
import re

arquivo = 'jk_couto_2ed.pdf'
loader = PyMuPDFLoader(f"/content/{arquivo}")
all_pages = loader.load()

print(f"Total de páginas carregadas: {len(all_pages)}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f" Using device: {device}")

In [3]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": device}
)

text_splitter = SemanticChunker(
    embeddings=embeddings_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70
)

chunks = text_splitter.split_documents(all_pages)
print(f"Foram criados {len(chunks)} chunks")

vector_store = Chroma.from_documents(chunks, embeddings_model)
print("Base vetorial criada!")

In [13]:
from huggingface_hub import login
from google.colab import userdata
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HuggingFaceKey')

login(token=userdata.get('HuggingFaceKey'))

In [15]:
from sentence_transformers import CrossEncoder
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')

MODEL_ID = "google/gemma-3-4b-it"

local_llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    pipeline_kwargs=dict(
        do_sample=True,
        max_new_tokens=2048,
        return_full_text=False
    )
)
chat_model = ChatHuggingFace(llm=local_llm)

In [18]:
from langchain_core.prompts import PromptTemplate

query = "Como o governo de JK lidou com a oposição política durante a construção de Brasília?"

# Como definido nos outros testes
bm25 = BM25Retriever.from_documents(chunks, k=5)
mmr = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 5, 'fetch_k': 30, 'lambda_mult': 0.6}
)
ensemble = EnsembleRetriever(retrievers=[bm25, mmr], weights=[0.5, 0.5])

# Prompt simples, o foco não é testar ele aqui!
prompt = PromptTemplate(
    input_variables=["contexto", "pergunta"],
    template="""
Use apenas o contexto abaixo para responder.

<contexto>
{contexto}
</contexto>

Pergunta: {pergunta}

Resposta:
"""
)

## Primeiro caso: SEM re-ranking

In [21]:
print("\n" + "="*80)
print("SEM RE-RANKING")
print("="*80)

docs_no_rerank = ensemble.invoke(query)

print(f"\nRecuperou(retrieved): {len(docs_no_rerank)} documentos")
print("\nTop 5 chunks:")
for i, doc in enumerate(docs_no_rerank[:10]):
    print(f"\n[{i+1}] {len(doc.page_content)} chars")
    print(doc.page_content[:250] + "...")

contexto_no = "\n---\n".join(d.page_content for d in docs_no_rerank[:5])
prompt_no = prompt.format(contexto=contexto_no, pergunta=query)

print("\n" + "="*80)
print("GERANDO RESPOSTA COM A LLM")
print("="*80)

response_no = chat_model.invoke([("user", prompt_no)])
print("\nResposta SEM re-ranking:")
print(response_no.content)

Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



SEM RE-RANKING

Recuperou(retrieved): 10 documentos

Top 5 chunks:

[1] 465 chars
Em 
termos práticos, resultou em muito pouco durante o mandato de JK. Mas foi fundamental à criação do Banco Interamericano de Desen­
volvimento (BID), em dezembro de 1959, e, no governo John Kennedy 
(1961-1963), da Aliança para o Progresso. A ação ...

[2] 321 chars
238 • JK
Fazer política no Brasil era, de um lado, a adesão à ditadura ou então, de 
todos os outros lados, inclusive o de dentro, a arte de fazer tudo o que era 
possível pela democratização. Anotação de JK no diário, em 8 de junho de 1974:
“Cumpri ...

[3] 581 chars
Muita 
conta para pagar, desequilíbrios importantes. Como a escalada das 
despesas públicas decorrente da concretização do Programa de Metas, 
especialmente com a construção de Brasília, os investimentos maciços 
em energia e transportes e a sustenta...

[4] 162 chars
JK • 227
Pane
Justiça se faça: a consolidação de Brasília como cidade e como capi­
tal federal deu-se no regim

## Segundo caso: Re-ranking com Cross-Encoder

Um modelo cross-encoder reavalia os documentos retornados, atribuindo uma pontuação para cada par (query, documento).


> Um cross-encoder é um tipo de modelo usado para medir o quão relevante um texto é para outro, normalmente uma pergunta e um documento.


Os resultados são então ordenados por relevância, retornando apenas os mais bem classificados, com maior precisão semântica.

In [20]:
rerank_ks = [3, 5, 8]

for k_final in rerank_ks:
    print("\n" + "="*80)
    print(f"COM RE-RANKING (K={k_final})")
    print("="*80)

    docs_initial = ensemble.invoke(query)

    # Re-rank padrão
    # Conecta par de query + chunk
    pairs = [[query, doc.page_content] for doc in docs_initial]

    # Pontua cada um
    scores = cross_encoder.predict(pairs)

    # Ordena com base em cada pontuação
    scored_docs = sorted(zip(docs_initial, scores), key=lambda x: x[1], reverse=True)
    docs_reranked = [doc for doc, score in scored_docs[:k_final]]

    print(f"\n top {k_final} documents ranqueados:")
    for i, (doc, score) in enumerate(scored_docs[:k_final]):
        print(f"\n[{i+1}] Score: {score:.4f} | {len(doc.page_content)} chars")
        print(doc.page_content[:250] + "...")

    contexto_yes = "\n---\n".join(d.page_content for d in docs_reranked)
    prompt_yes = prompt.format(contexto=contexto_yes, pergunta=query)

    print("\n" + "="*80)
    print("GERANDO RESPOSTA COM A LLM...")
    print("="*80)

    response_yes = chat_model.invoke([("user", prompt_yes)])
    print(f"\n Resposta COM re-ranking (K={k_final}):")
    print(response_yes.content)
    print("\n" + "="*80)

Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



COM RE-RANKING (K=3)

 top 3 documents ranqueados:

[1] Score: -2.1041 | 162 chars
JK • 227
Pane
Justiça se faça: a consolidação de Brasília como cidade e como capi­
tal federal deu-se no regime militar. Reconheceram a obra, mas não 
o fundador....

[2] Score: -2.4372 | 340 chars
24 • JK
de São Francisco, realizações de Juscelino quando prefeito da capital 
mineira, precursoras da construção de Brasília. Mas o evento foi, para 
nós, um grande sucesso de público, de crítica e para a estrutura física 
da UNE. Os recursos libera...

[3] Score: -2.5010 | 737 chars
JK • 239
Nessa época, a ABL negociava financiamento de instituição federal 
para a construção de prédio de 29 andares no terreno anexo ao de sua 
sede, doado pelo governo federal, na avenida Presidente Wilson, cen­
tro do Rio de Janeiro. Projeto impo...

GERANDO RESPOSTA COM A LLM...


Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Resposta COM re-ranking (K=3):
O governo de JK lidou com a oposição política durante a construção de Brasília de forma a evitar que a eleição de JK pudesse ser vista como um desafio ao governo militar. Diversas correntes de dentro e de fora se uniram em torno da terceira tentativa do escritor goiano Bernardo Élis para que ele fosse reconhecido.


COM RE-RANKING (K=5)

 top 5 documents ranqueados:

[1] Score: -2.1041 | 162 chars
JK • 227
Pane
Justiça se faça: a consolidação de Brasília como cidade e como capi­
tal federal deu-se no regime militar. Reconheceram a obra, mas não 
o fundador....

[2] Score: -2.4372 | 340 chars
24 • JK
de São Francisco, realizações de Juscelino quando prefeito da capital 
mineira, precursoras da construção de Brasília. Mas o evento foi, para 
nós, um grande sucesso de público, de crítica e para a estrutura física 
da UNE. Os recursos libera...

[3] Score: -2.5010 | 737 chars
JK • 239
Nessa época, a ABL negociava financiamento de instituição federal 
para a

Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Resposta COM re-ranking (K=5):
O contexto indica que a popularidade de JK incomodava a oposição, e para "prová-la", o governo militar procurou impedi-lo de ser eleito, apoiando a terceira tentativa do escritor goiano Bernardo Élis como candidato. Isso demonstra uma forma de lidar com a oposição, buscando alternativas para diminuir seu poder e influência.


COM RE-RANKING (K=8)

 top 8 documents ranqueados:

[1] Score: -2.1041 | 162 chars
JK • 227
Pane
Justiça se faça: a consolidação de Brasília como cidade e como capi­
tal federal deu-se no regime militar. Reconheceram a obra, mas não 
o fundador....

[2] Score: -2.4372 | 340 chars
24 • JK
de São Francisco, realizações de Juscelino quando prefeito da capital 
mineira, precursoras da construção de Brasília. Mas o evento foi, para 
nós, um grande sucesso de público, de crítica e para a estrutura física 
da UNE. Os recursos libera...

[3] Score: -2.5010 | 737 chars
JK • 239
Nessa época, a ABL negociava financiamento de instituição feder

# Observações finais

Eu achei que com re-ranking, me deu uma resposta mais lúcida e tenho uma premissas em torno disso: com mais chunks, a SLM(foco nela) pode se embananar mais com muito conteúdo, por isso que o foco é filtrar os MAIS importantes.

A escolha que eu achei interessante foi top 3, poucos chunks e não diminui o resultado dado.